In [3]:
import os

from smolagents import ToolCollection, OpenAIServerModel, ToolCallingAgent, Tool, ToolCallingAgent, ActionStep, PromptTemplates, MultiStepAgent
from mcp import StdioServerParameters

os.environ['NEBIUS_API_KEY'] = open('secret.txt', 'r').read().strip()

REPO = "/mnt/c/Users/Uniholder/Git/biotools-mcp-server"

server = StdioServerParameters(
    command="node",
    args=[f"{REPO}/build/index.js", "stdio"]
)

In [11]:
MODEL_T = "Qwen/Qwen3-235B-A22B-Thinking-2507"
MODEL_I = "Qwen/Qwen3-235B-A22B-Instruct-2507"

model_t = OpenAIServerModel(
    model_id=MODEL_T,
    api_key=os.environ["NEBIUS_API_KEY"],
    api_base="https://api.studio.nebius.com/v1/",
    temperature=0,
)

model_i = OpenAIServerModel(
    model_id=MODEL_I,
    api_key=os.environ["NEBIUS_API_KEY"],
    api_base="https://api.studio.nebius.com/v1/",
    temperature=0,
)

In [4]:
with ToolCollection.from_mcp(
        server_parameters=server,
        trust_remote_code=True,
        structured_output=False
    ) as tools:
        # Проверить, что тулы действительно подхватились:
        print([t.name for t in tools.tools])

['search_pubmed', 'get_publication_details', 'get_publication_abstract', 'search_uniprot', 'get_protein_entry', 'get_protein_sequence', 'get_nucleotide_sequence', 'compare_annotations', 'find_intron_exons', 'align_promoters', 'get_cross_references', 'analyze_ptms', 'get_pathway_data', 'analyze_gc_content', 'find_restriction_sites', 'predict_orfs', 'assemble_fragments', 'predict_protein_properties', 'predict_transmembrane_regions', 'scan_protein_motifs', 'blast_search', 'psi_blast_search', 'align_sequences_global', 'align_sequences_local', 'generate_dotplot', 'multiple_sequence_alignment', 'highlight_conserved_regions', 'generate_sequence_logo', 'export_alignment', 'get_protein_structure', 'analyze_secondary_structure', 'predict_rna_secondary_structure', 'scan_rna_motifs', 'build_phylogenetic_tree', 'compare_phylogenetic_trees', 'log_analysis_parameters', 'generate_resource_map']


In [5]:
def check_tool(tool, **kwargs):
    with ToolCollection.from_mcp(server, trust_remote_code=True, structured_output=False) as tc:
        ex = [t for t in tc.tools if t.name==tool][0]
        out = ex(kwargs if kwargs else {})
        print(out)

In [21]:
def run_query(user_prompt, model, max_steps=10):
    with ToolCollection.from_mcp(
        server_parameters=server,
        trust_remote_code=True,
        structured_output=False
    ) as tools:
        agent = ToolCallingAgent(
            model=model,
            tools=[*tools.tools],
            add_base_tools=False,
            max_steps=max_steps,
        )
        # agent.prompt_templates["system_prompt"] = SYSTEM_PROMPT
        return agent.run(user_prompt)

In [24]:
gene_or_protein = 'GHR'

user_prompt = f"""
Find publications with description of gene|protein {gene_or_protein} functionality. 
If they show relation with aging, longevity, severe diseases prioritize this.

Return summary based ONLY on the results of your search.
Give article IDs for reference.
"""

In [25]:
output = run_query(user_prompt, model_i)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find publications with description of gene|protein GHR functionality.                                           │
│ If they show relation with aging, longevity, severe diseases prioritize this.                                   │
│                                                                                                                 │
│ Return summary based ONLY on the results of your search.                                                        │
│ Give article IDs for reference.                                                                                 │
│                                                                                                                 │
╰─ OpenAIServerModel - Qwen/Qwen3-235B-A22B-Instruct-2507 ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'search_pubmed' with arguments: {'term': 'GHR gene protein aging longevity disease',              │
│ 'max_results': 10}                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Found 10 paper(s) for search term: 'GHR gene protein aging longevity disease'

PMID: 37561089
Title: Genes That Extend Lifespan May Do So by Mitigating the Increased Risk of Death Posed by Having Hypertension.
Authors: Morris BJ, Donlon TA
Journal: American journal of hypertension
Publication Date: 2023 Nov 15
DOI: doi: 10.1093/ajh/hpad070
---
PMID: 35952979
Title: Disruption of Growth Hormone Receptor in Adipocytes Improves Insulin Sensitivity and Lifespan in Mice.
Authors: List EO, Berryman DE, Slyby J, et al.
Journal: Endocrinology
Publication Date: 2022 Oct 1
DOI: doi: 10.1210/endocr/bqac129
---
PMID: 34074802
Title: Association of growth hormone receptor gene variant with longevity in men is due to amelioration of 
increased mortality risk from hypertension.
Authors: Donlon TA, Chen R, Masaki KH, et al.
Journal: Aging
Publication Date: 2021 Jun 1
DOI: doi: 10.18632/aging.203133
---
PMID: 33755309
Title: Physiological and metabolic characteristics of novel double-mutant female mice with targeted disruption of 
both growth hormone-releasing hormone and growth hormone receptor.
Authors: Icyuz M, Zhang F, Fitch MP, et al.
Journal: Aging cell
Publication Date: 2021 Apr
DOI: doi: 10.1111/acel.13339
---
PMID: 33031577
Title: Data mining of human plasma proteins generates a multitude of highly predictive aging clocks that reflect 
different aspects of aging.
Authors: Lehallier B, Shokhirev MN, Wyss-Coray T, et al.
Journal: Aging cell
Publication Date: 2020 Nov
DOI: doi: 10.1111/acel.13256
---
PMID: 28528685
Title: IGF-I deficiency, longevity and cancer protection of patients with Laron syndrome.
Authors: Laron Z, Kauli R, Lapkina L, et al.
Journal: Mutation research. Reviews in mutation research
Publication Date: 2017 Apr-Jun
DOI: doi: 10.1016/j.mrrev.2016.08.002
---
PMID: 28425851
Title: Altered structure and function of adipose tissue in long-lived mice with growth hormone-related mutations.
Authors: Darcy J, McFadden S, Bartke A
Journal: Adipocyte
Publication Date: 2017 Apr 3
DOI: doi: 10.1080/21623945.2017.1308990
---
PMID: 28299640
Title: The role of transplanted visceral fat from the long-lived growth hormone receptor knockout mice on insulin 
signaling.
Authors: Bennis MT, Schneider A, Victoria B, et al.
Journal: GeroScience
Publication Date: 2017 Feb
DOI: doi: 10.1007/s11357-017-9957-y
---
PMID: 28073935
Title: Brain Structure and Function Associated with Younger Adults in Growth Hormone Receptor-Deficient Humans.
Authors: Nashiro K, Guevara-Aguirre J, Braskie MN, et al.
Journal: The Journal of neuroscience : the official journal of the Society for Neuroscience
Publication Date: 2017 Feb 15
DOI: doi: 10.1523/JNEUROSCI.1929-16.2016
---
PMID: 27208894
Title: Enhanced Cognition and Hypoglutamatergic Signaling in a Growth Hormone Receptor Knockout Mouse Model of 
Successful Aging.
Authors: Hascup KN, Lynn MK, Fitzgerald PJ, et al.
Journal: The journals of gerontology. Series A, Biological sciences and medical sciences
Publication Date: 2017 Mar 1
DOI: doi: 10.1093/gerona/glw088

[Step 1: Duration 4.08 seconds| Input tokens: 14,284 | Output tokens: 34]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The GHR (growth hormone receptor) gene and protein are │
│ strongly linked to aging, longevity, and protection from severe diseases based on multiple studies. Key         │
│ findings from the literature include:\n\n1. **PMID 34074802**: A study on elderly American men of Japanese      │
│ ancestry found that a specific variant (rs4130113) in the GHR gene is associated with longevity, particularly   │
│ by reducing mortality risk in individuals with hypertension. The longevity-associated genotype appears to       │
│ enhance cellular resilience mechanisms, allowing hypertensive men with this variant to live as long as          │
│ normotensive individuals.\n\n2. **PMID 35952979**: Disruption of the GHR in adipocytes (fat cells) in mice      │
│ improved insulin sensitivity, reduced frailty, and extended lifespan, especially in males. Approximately 23% of │
│ the lifespan extension seen in full GHR knockout mice was attributed to GHR disruption in adipose tissue.\n\n3. │
│ **PMID 37561089**: Research indicates that longevity-associated variants in the GHR gene mitigate the increased │
│ mortality risk posed by hypertension, suggesting a role in enhancing cellular resilience in the context of      │
│ cardiometabolic stress.\n\n4. **PMID 33755309**: Female mice with double knockout of GHR and growth             │
│ hormone-releasing hormone (GHRH) showed enhanced insulin sensitivity, reduced body weight, and signs of         │
│ extended lifespan, indicating that GHR disruption contributes to metabolic health and longevity.\n\n5. **PMID   │
│ 28528685**: Humans with Laron syndrome, caused by GHR gene mutations leading to IGF-I deficiency, exhibit not   │
│ only dwarfism and obesity but also remarkable protection from cancer and diabetes, and display extended         │
│ healthspan and longevity.\n\nIn summary, GHR functionality is critically involved in aging and longevity, with  │
│ its disruption or genetic variation conferring protection against age-related diseases such as hypertension,    │
│ diabetes, and cancer, likely through improved metabolic regulation and enhanced cellular resilience.'}          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The GHR (growth hormone receptor) gene and protein are strongly linked to aging, longevity, and 
protection from severe diseases based on multiple studies. Key findings from the literature include:

1. **PMID 34074802**: A study on elderly American men of Japanese ancestry found that a specific variant 
(rs4130113) in the GHR gene is associated with longevity, particularly by reducing mortality risk in individuals 
with hypertension. The longevity-associated genotype appears to enhance cellular resilience mechanisms, allowing 
hypertensive men with this variant to live as long as normotensive individuals.

2. **PMID 35952979**: Disruption of the GHR in adipocytes (fat cells) in mice improved insulin sensitivity, reduced
frailty, and extended lifespan, especially in males. Approximately 23% of the lifespan extension seen in full GHR 
knockout mice was attributed to GHR disruption in adipose tissue.

3. **PMID 37561089**: Research indicates that longevity-associated variants in the GHR gene mitigate the increased 
mortality risk posed by hypertension, suggesting a role in enhancing cellular resilience in the context of 
cardiometabolic stress.

4. **PMID 33755309**: Female mice with double knockout of GHR and growth hormone-releasing hormone (GHRH) showed 
enhanced insulin sensitivity, reduced body weight, and signs of extended lifespan, indicating that GHR disruption 
contributes to metabolic health and longevity.

5. **PMID 28528685**: Humans with Laron syndrome, caused by GHR gene mutations leading to IGF-I deficiency, exhibit
not only dwarfism and obesity but also remarkable protection from cancer and diabetes, and display extended 
healthspan and longevity.

In summary, GHR functionality is critically involved in aging and longevity, with its disruption or genetic 
variation conferring protection against age-related diseases such as hypertension, diabetes, and cancer, likely 
through improved metabolic regulation and enhanced cellular resilience.

Final answer: The GHR (growth hormone receptor) gene and protein are strongly linked to aging, longevity, and 
protection from severe diseases based on multiple studies. Key findings from the literature include:

1. **PMID 34074802**: A study on elderly American men of Japanese ancestry found that a specific variant 
(rs4130113) in the GHR gene is associated with longevity, particularly by reducing mortality risk in individuals 
with hypertension. The longevity-associated genotype appears to enhance cellular resilience mechanisms, allowing 
hypertensive men with this variant to live as long as normotensive individuals.

2. **PMID 35952979**: Disruption of the GHR in adipocytes (fat cells) in mice improved insulin sensitivity, reduced
frailty, and extended lifespan, especially in males. Approximately 23% of the lifespan extension seen in full GHR 
knockout mice was attributed to GHR disruption in adipose tissue.

3. **PMID 37561089**: Research indicates that longevity-associated variants in the GHR gene mitigate the increased 
mortality risk posed by hypertension, suggesting a role in enhancing cellular resilience in the context of 
cardiometabolic stress.

4. **PMID 33755309**: Female mice with double knockout of GHR and growth hormone-releasing hormone (GHRH) showed 
enhanced insulin sensitivity, reduced body weight, and signs of extended lifespan, indicating that GHR disruption 
contributes to metabolic health and longevity.

5. **PMID 28528685**: Humans with Laron syndrome, caused by GHR gene mutations leading to IGF-I deficiency, exhibit
not only dwarfism and obesity but also remarkable protection from cancer and diabetes, and display extended 
healthspan and longevity.

In summary, GHR functionality is critically involved in aging and longevity, with its disruption or genetic 
variation conferring protection against age-related diseases such as hypertension, diabetes, and cancer, likely 
through improved metabolic regulation and enhanced cellular resilience.

[Step 5: Duration 10.20 seconds| Input tokens: 101,932 | Output tokens: 806]

In [23]:
print(output)

Based on the search results, here is a summary of publications describing APOE gene/protein functionality with emphasis on aging, longevity, and severe diseases:

1. **PMID: 41060400** - This study investigates the role of APOE in cerebrovascular lipid metabolism in Alzheimer's disease (AD). The APOE ε4 allele, the strongest genetic risk factor for AD, is associated with significant alterations in cerebrovascular lipid profiles. Specifically, APOE ε4 correlates with increased phosphatidylethanolamine (PE) and decreased sphingomyelin (SM) levels. These lipid changes are linked to amyloid-β deposition and tau pathology, suggesting that APOE-mediated lipid dysregulation contributes to AD pathogenesis. This highlights APOE's critical role in age-related neurodegenerative processes.

2. **PMID: 41014658** - A review discussing gene therapy targets for aging-associated diseases, including the role of APOE variants in protein homeostasis. The paper emphasizes APOE as a key target in combating